In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time, timedelta
from SynthSpread.spreadviewer_class import SpreadSingle, SpreadViewerData, norm_coeff
from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Strategies.MultipleMarketsIntensity_class import MultiTradeIntensity as TI
from Math.ti_class import TI_class, VI_class, TR_class
from Math.lm_class import kalman, LinearModel
from Math.accumfeatures import EMA, MA, MSTD, DifferentialEMA, DerivativeEMA
from Strategies.IntensityHawkes_strategy.model_class import HawkesIntensity
from Strategies.IntensityHawkes_strategy.backtest_class import BacktestIB
from Strategies.IntensityHawkes_strategy.strategy_class import StrategyHI, VolumeClass
tol=(1e-1)/2

# Data Loading

In [3]:
params_dict = {}

params_dict['tenor_list'] = ['m']
params_dict['tn1_list'] = [1]
params_dict['mkt_list'] = ['de'] * len(params_dict['tenor_list'])
params_dict['tn2_list'] = []
params_dict['prod'] = 'base'
params_dict['venue_list'] = ['eex']*len(params_dict['mkt_list'])
params_dict['start_date'] = datetime(2024, 6, 20)
params_dict['end_date'] = datetime(2024, 6, 20)
params_dict['ns'] = 2

# Fetch trades and best orders for the curve
assembler = TPDataAssembly(source='trayport', user='matej')
# assembler.set_start_end_time(start=[10,0,0], end=[12,0,0])    
trades_dict = assembler.get_data(params_dict, target_data='trades')
#assembler.set_data_source('database')
ba_dict = assembler.get_data(params_dict, target_data='best_orders')

assert ba_dict.keys() == trades_dict.keys(), "Curve doesn't fit for both trades and best_orders"

trades = pd.DataFrame()
ba = pd.DataFrame()
products = []
for key in trades_dict.keys():
    ba_aux = ba_dict[key].copy()
    trade_aux = trades_dict[key].copy()
    trade_aux.columns = [a + '_' + key for a in trade_aux.columns]
    ba_aux.columns = [a + '_' + key for a in ba_aux.columns]
    if trades.empty:
        trades = trade_aux.copy()
    else:
        trades = pd.concat([trades, trade_aux])
    if ba.empty:
        ba = ba_aux.copy()
    else:
        ba = pd.concat([ba, ba_aux])
    products.append(key)
trades.sort_index(inplace=True)
ba.sort_index(inplace=True)
ba.index.name='datetime'

ti_inst = TI(trades, ba, products)

ti_inst.prepare_data()


data_raw = ti_inst.data

# data = data_raw[data_raw['broker_id_dew1']==1441][['price_dew1', 'volume_dew1','bidbestprice_dew1',
#                   'askbestprice_dew1', 'mid_dew1', 'trade_side_dew1']].copy()

data = data_raw[['price_dem1', 'volume_dem1','bidbestprice_dem1',
                   'askbestprice_dem1', 'mid_dem1', 'trade_side_dem1']].copy()
    
data.columns = [a.split('_')[0] for a in data.columns]
data.columns = ['trd_price', 'volume', 'bid_price', 'ask_price', 'mid_price', 'trd_side']

data.to_csv(r's:\Algo\Files\andrej\Data\int_data_lead_production_sample.csv')

https://referencedata.trayport.com/instruments
Duration: 2.142s


KeyboardInterrupt: 

# Parsing calibration results simple

In [7]:
# Load the pickled object
import pickle
with open(r'out_dict_deq1_dem1_07032024.pkl', 'rb') as file:
    loaded_object = pickle.load(file)

df=pd.DataFrame([(k[0],k[1],k[2], k[3], k[4], v) for k,v in loaded_object.items()], columns=['param_max_secs_between_trades',
                                                                                          'param_cluster_trade_num_threshold',
                                                                                          'param_min_price_movement',
                                                                                          'param_take_profit',
                                                                                          'param_stop_loss',
                                                                                          
                                                                                           'cumpnl']).sort_values(by=['cumpnl'], ascending=False)

In [8]:
filter1=df['param_min_price_movement']>0.1
filter2=df['param_take_profit']>=df['param_stop_loss'].apply(abs)
filter3=df['param_max_secs_between_trades']<45

filter=filter1&filter2&filter3


df[filter][0:50]

,param_max_secs_between_trades,param_cluster_trade_num_threshold,param_min_price_movement,param_take_profit,param_stop_loss,cumpnl
335,4,0.009,2.25,10,False,2.245
334,4,0.009,2.25,10,True,2.245
333,4,0.009,2.00,10,False,-0.345
332,4,0.009,2.00,10,True,-0.345
76,1,0.009,1.50,10,True,-0.490
77,1,0.009,1.50,10,False,-0.490
330,4,0.009,1.75,10,True,-0.895
331,4,0.009,1.75,10,False,-0.895
328,4,0.009,1.50,10,True,-1.135
329,4,0.009,1.50,10,False,-1.135


# Parsing calibration results new

## dew1_dem1

### Fixed Take profit

In [40]:
# Load the pickled object
import pickle
with open(r's:\Algo\Files\andrej\LeadLag\out_dict_dew1_dem1_20240621.pkl', 'rb') as file:
    loaded_object = pickle.load(file)

df=pd.DataFrame([(k[0],k[1],k[2], k[3], k[4], v[0],v[1],v[2], v[3], v[4]) for k,v in loaded_object.items()], columns=['MSBT',
                                                                                          'CTNM',
                                                                                          'MPM',
                                                                                          'TP',
                                                                                          'SL',
                                                                                          
                                                                                           'cumpnl', 'sharp', 'max_drawdown', 'num_trades', 'trade_per_day']).sort_values(by=['sharp'], ascending=False)

In [41]:
filter1=df['MPM']>0.1
filter2=True#df['TP']>=df['SL'].apply(abs)
filter3=df['num_trades']>40

filter=filter1&filter2&filter3


df[filter][0:50]

,MSBT,CTNM,MPM,TP,SL,cumpnl,sharp,max_drawdown,num_trades,trade_per_day
3344,10,9,0.2,1.00,-0.40,5.095,2.786019,3.1950,62,1.441860
3345,10,9,0.2,1.00,-0.50,4.045,2.748145,4.0550,60,1.395349
3346,10,9,0.2,1.00,-0.75,3.020,2.660105,3.7550,54,1.255814
3347,10,9,0.2,1.00,-1.00,1.275,2.461024,4.4900,52,1.209302
6981,30,17,0.3,1.00,-0.50,2.735,2.412272,2.7750,50,1.162791
3340,10,9,0.2,0.75,-0.75,3.170,2.339937,3.5000,56,1.302326
6947,30,17,0.2,1.00,-1.00,8.660,2.325259,6.2950,124,2.883721
3338,10,9,0.2,0.75,-0.40,4.945,2.324270,3.2300,64,1.488372
3339,10,9,0.2,0.75,-0.50,3.670,2.282473,3.8950,62,1.441860
3343,10,9,0.2,1.00,-0.30,1.670,2.244264,3.9400,62,1.441860


### With action closing

In [43]:
# Load the pickled object
import pickle
with open(r's:\Algo\Files\andrej\LeadLag\out_dict_dew1_dem1_20240624.pkl', 'rb') as file:
    loaded_object = pickle.load(file)

df=pd.DataFrame([(k[0],k[1],k[2], k[3], k[4], v[0],v[1],v[2], v[3], v[4]) for k,v in loaded_object.items()], columns=['MSBT',
                                                                                          'CTNM',
                                                                                          'MPM',
                                                                                          'TP',
                                                                                          'SL',
                                                                                          
                                                                                           'cumpnl', 'sharp', 'max_drawdown', 'num_trades', 'trade_per_day']).sort_values(by=['cumpnl'], ascending=False)

In [44]:
filter1=df['MPM']>=0.1
filter2=df['SL']<-0.1
filter3=df['MSBT']<100

filter=filter1&filter2&filter3


df[filter][0:50]

,MSBT,CTNM,MPM,TP,SL,cumpnl,sharp,max_drawdown,num_trades,trade_per_day
0,10,9,0.2,1.0,-0.4,3.765,2.707579,3.76,62,1.44186


## deq1_dem1

### Fixed Take profit

In [20]:
# Load the pickled object
import pickle
with open(r'out_dict_deq1_dem1_08032024.pkl', 'rb') as file:
    loaded_object = pickle.load(file)

df=pd.DataFrame([(k[0],k[1],k[2], k[3], k[4], v[0],v[1],v[2], v[3], v[4]) for k,v in loaded_object.items()], columns=['MSBT',
                                                                                          'CTNM',
                                                                                          'MPM',
                                                                                          'TP',
                                                                                          'SL',
                                                                                          
                                                                                           'cumpnl', 'sharp', 'max_drawdown', 'num_trades', 'trade_per_day']).sort_values(by=['cumpnl'], ascending=False)

In [21]:
filter1=df['MPM']>0.0
filter2=df['TP']>=df['SL'].apply(abs)
filter3=df['MSBT']<100

filter=filter1&filter2&filter3


df[filter][0:50]

,MSBT,CTNM,MPM,TP,SL,cumpnl,sharp,max_drawdown,num_trades,trade_per_day
4546,30,3,0.10,2.00,-0.75,27.060,0.436153,11.2475,198,4.829268
7045,50,7,0.10,2.00,-0.50,25.480,2.368636,9.1950,230,5.609756
5636,40,3,0.05,1.00,-0.75,24.985,1.176155,6.4275,304,7.414634
7948,60,3,0.20,2.00,-1.50,24.465,0.987591,5.9800,130,3.170732
7895,60,3,0.05,2.00,-0.50,24.060,2.138794,9.5650,308,7.512195
5637,40,3,0.05,1.00,-1.00,24.055,1.534391,6.9550,274,6.682927
4549,30,3,0.10,2.00,-2.00,23.740,0.888648,9.3200,132,3.219512
6772,50,3,0.05,2.00,-1.00,23.580,1.221613,7.5550,218,5.317073
6823,50,3,0.20,2.00,-1.50,23.390,1.104941,4.6800,122,2.975610
8286,60,9,0.10,1.00,-0.75,23.375,1.919319,4.6600,258,6.292683


### Trailing Take profit

In [22]:
# Load the pickled object
import pickle
with open(r'out_dict_deq1_dem1_trailing_tp_13032024.pkl', 'rb') as file:
    loaded_object = pickle.load(file)

df=pd.DataFrame([(k[0],k[1],k[2], k[3], k[4], v[0],v[1],v[2], v[3], v[4]) for k,v in loaded_object.items()], columns=['MSBT',
                                                                                          'CTNM',
                                                                                          'MPM',
                                                                                          'TP',
                                                                                          'SL',
                                                                                          
                                                                                           'cumpnl', 'sharp', 'max_drawdown', 'num_trades', 'trade_per_day']).sort_values(by=['cumpnl'], ascending=False)

In [23]:
filter1=df['MPM']>0.0
filter2=df['TP']>=df['SL'].apply(abs)
filter3=df['MSBT']<100

filter=filter1&filter2&filter3


df[filter][0:50]

,MSBT,CTNM,MPM,TP,SL,cumpnl,sharp,max_drawdown,num_trades,trade_per_day
5637,50,7,0.10,1.0,-0.50,23.515,2.064483,9.1200,252,6.146341
6317,60,3,0.05,1.0,-0.50,21.545,2.392419,9.8300,350,8.536585
5718,50,9,0.05,1.0,-0.75,20.530,1.083311,7.9000,220,5.365854
5638,50,7,0.10,1.0,-0.75,19.405,2.869682,9.2250,228,5.560976
5179,40,15,0.30,1.0,-1.00,18.785,2.252047,2.9850,60,1.463415
6711,60,11,0.05,0.5,-0.30,18.510,1.734927,4.5450,384,9.365854
5612,50,7,0.05,0.5,-0.50,17.715,1.820608,13.0600,378,9.219512
4717,40,7,0.05,1.0,-0.50,15.910,1.912262,8.8650,270,6.585366
6731,60,11,0.10,0.5,-0.30,15.860,2.069982,4.4250,334,8.146341
5436,50,3,0.10,1.0,-0.30,15.435,1.207507,7.5000,394,9.609756
